# VT_Preprocessing

Variable Tone preprocessing: raw photometry and video tracking → one merged CSV per mouse.

**Pipeline:** **1. VT_Preprocessing** → 2. VT_Publication_Analysis → 3. VT_Publication_Figure / 4. VT_SK3_Rescue_Figure

### Inputs (read from this repo)
| Path | Contents |
|---|---|
| `Photometry/VariableTone/RWD/RWD_Fluorescence/` | RWD photometry exports (410 / 470 nm, TTL events), 24 mice |
| `Photometry/VariableTone/Bonsai/Raw/` | Raw Bonsai tracking and TTL, 23 mice (F19 has no video) |

### Outputs (downloaded as a zip at the end)
- One merged CSV per mouse (`…_merged.csv`, or `…_merged_NOBONSAIDATA.csv` without video), for `Photometry/VariableTone/RWD/RWD_Processed/`
- `preprocessing_qc.csv` and a QC figure per mouse

### How to run
Run all cells top to bottom. `SKIP_EXISTING=True` skips mice already in `RWD_Processed`; set it to `False` to rebuild all 24 from raw. `VERIFY_VS_EXISTING=True` compares each rebuilt file with a reference version of the processed data. The run is pinned to the current commit of `main`, which is printed.

### Stages
- **A** Clean Bonsai: scale tracking, collapse duplicated frames, decode TTL pulses (100 ms tone, 500 ms dispense, 1000 ms retrieval)
- **B** Photometry: double-exponential debleaching → motion correction against 410 nm → z-score relative to the lowest 25% of samples (`Corrected_Fluorescence_Z_25`)
- **C** Events: tone onsets by duration (`ToneStart_5000/10000/20000`), dispenses and retrievals from the photometry TTL record
- **D** Align the Bonsai clock to the photometry clock and interpolate tracking onto it
- **E** Position: locate the feeder from retrieval positions, convert to cm and rotate to the arena axes (`DistFromFED`)
- **E2** Blank samples where the tracker lost the mouse (impossible jumps to a fixed spot); flagged in `TrackingLost`
- **F** Run all mice, QC, verify, save

In [ ]:
# @title Setup — imports
import io, os, re, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.optimize import curve_fit
from scipy.stats import linregress
from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm
print("Imports OK.")


Imports OK.


In [ ]:
# @title Output paths & run options
# Inputs are read from the GitHub repo; outputs are written to the Colab runtime
# and downloaded at the end. Commit the outputs to the repo yourself.

OUT_DIR = '/content/VT_Preprocessing_out'   # Colab-local scratch, zipped at the end
QC_DIR  = f'{OUT_DIR}/QC'                   # per-animal QC figures
os.makedirs(QC_DIR, exist_ok=True)

# Skip animals whose merged CSV is already committed to GitHub (RWD_Processed).
# True = the finished "run once ever" state: with the processed data in the repo
# this skips all 24 and the notebook is a documented no-op.
# Set False to force a full rebuild from raw (what you do to reproduce/verify).
SKIP_EXISTING = True

# Compare each freshly processed animal against the ORIGINAL merged data, pinned
# to the commit before the processed data was replaced (see LEGACY_REF below).
VERIFY_VS_EXISTING = True

print(f"Output -> {OUT_DIR}\nQC     -> {QC_DIR}\n"
      f"SKIP_EXISTING={SKIP_EXISTING}  VERIFY_VS_EXISTING={VERIFY_VS_EXISTING}")


Output -> /content/VT_Preprocessing_out
QC     -> /content/VT_Preprocessing_out/QC
SKIP_EXISTING=True  VERIFY_VS_EXISTING=True


In [ ]:
# @title Parameters & configuration
GITHUB_OWNER = "CreedLab-Neuro"   # or "MasBarr"
GITHUB_REPO  = "Manuscript_PMID41279070"
GH_API  = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
GH_BASE = f"{GH_API}/contents"
VT      = "Photometry/VariableTone"

# ── Pin this run to one commit ────────────────────────────────────────────────
# raw.githubusercontent caches BRANCH urls (~5 min) and different edge nodes
# expire independently, so fetching ".../main/..." right after a data push can
# hand you a mix of old and new files -- silently, in the same run. Resolving
# main to its commit SHA once and fetching ".../<sha>/..." avoids this entirely:
# SHA urls are immutable, so a cache hit is always correct.
# It also makes the run self-documenting -- the printed SHA is exactly which
# version of the data produced these outputs.
DATA_SHA = requests.get(f"{GH_API}/commits/main").json()['sha']

API_SCALED    = f"{GH_BASE}/{VT}/Bonsai/Scaled_Renamed?ref={DATA_SHA}"
API_RAW       = f"{GH_BASE}/{VT}/Bonsai/Raw?ref={DATA_SHA}"
API_RWD_FLUOR = f"{GH_BASE}/{VT}/RWD/RWD_Fluorescence?ref={DATA_SHA}"
API_MERGED    = f"{GH_BASE}/{VT}/RWD/RWD_Processed?ref={DATA_SHA}"

# Reference version of RWD_Processed that VERIFY_VS_EXISTING compares against: a
# fixed earlier commit, so the check never compares the output with itself.
LEGACY_REF     = "43bee7b"
API_MERGED_REF = f"{GH_BASE}/{VT}/RWD/RWD_Processed?ref={LEGACY_REF}"

BONSAI_SOURCE = 'raw'   # 'raw' (canonical — clean from raw in-notebook) or 'scaled'

CONDITION_MAP = {
    'F1':'SNI','F26':'SNI','F29':'SNI','M1':'SNI','M3':'SNI','M11':'SNI','M12':'SNI','M13':'SNI',
    'F2':'SHAM','F3':'SHAM','F18':'SHAM','F19':'SHAM','M2':'SHAM','M7':'SHAM','M8':'SHAM','M9':'SHAM',
    'F32':'SK3','F33':'SK3','M21':'SK3','M23':'SK3','M24':'SK3','M26':'SK3','M27':'SK3','M28':'SK3',
}
C57_MICE = {'F1','F2','F3','M1','M2','M3'}

TONE_PULSE=(0.05,0.15); DISPENSE_PULSE=(0.4,0.7); RETRIEVAL_PULSE=(0.9,1.2)
TONE_DURATIONS=[5000,5000,5000,10000,20000,10000,10000,20000,5000,10000,
                10000,20000,20000,5000,20000,5000,10000,20000,5000,20000]
FED_X_CM=9.0

# Stage E thresholds
NEAR_FED_CM=8.0; SPEED_MAX=80.0; MIN_REG_PTS=100
R2_FLAG=0.70; MAE_FLAG=2.0; ANCHOR_COL='DistFromFED_orig'; OVERWRITE_DISTFROMFED=True

# Stage D column roles:
# - BONSAI_ALIGN: Bonsai events used to align the clock (proximity-matched to RWD; incl. tones)
# - BONSAI_SNAP : Bonsai binary columns written to the merged CSV (tones now come from RWD)
# - BONSAI_CONT : continuous tracking interpolated onto the RWD axis
BONSAI_ALIGN =['ToneStart_5000','ToneStart_10000','ToneStart_20000','Dispense_Bonsai','Retrieval_Bonsai']
BONSAI_SNAP  =['Dispense_Bonsai','Retrieval_Bonsai']
BONSAI_CONT  =['Xpos','Ypos','Speed_cm','DistFromFED']
RWD_EVENT_COLS=['ToneStart_5_RWD','ToneStart_10_RWD','ToneStart_20_RWD','Dispense_RWD','Retrieval_RWD']

def get_animal_id(s):
    m=re.search(r'([MF]\d+)', s); return m.group(1) if m else None
def get_animal_num(aid):
    m=re.search(r'\d+', aid or ''); return int(m.group()) if m else 999
print(f"Config ready. BONSAI_SOURCE = {BONSAI_SOURCE}")
print(f"Data version (main @ commit): {DATA_SHA[:10]}   legacy verify ref: {LEGACY_REF}")


In [ ]:
# @title GitHub loaders
def gh_list(api_url):
    r=requests.get(api_url); r.raise_for_status()
    return [{'name':f['name'],'download_url':f['download_url']}
            for f in r.json() if f['name'].endswith('.csv')]

def gh_read(url, **kw):
    r=requests.get(url); r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), low_memory=False, **kw)

def gh_load_folder(api_url, desc="Loading"):
    files=gh_list(api_url); out={}
    def _one(f): return get_animal_id(f['name']), f['name'], gh_read(f['download_url'])
    with ThreadPoolExecutor(max_workers=10) as ex:
        futs={ex.submit(_one,f):f['name'] for f in files}
        with tqdm(total=len(files), desc=desc) as pbar:
            for fut in as_completed(futs):
                try:
                    aid,name,df=fut.result()
                    if aid: out[aid]={'name':name,'df':df}
                except Exception as e: print(f"  x {futs[fut]}: {e}")
                pbar.update(1)
    return out
print("Loaders defined.")


Loaders defined.


In [ ]:
# @title Bonsai cleaner (clean_bonsai_raw)
# Cleans a raw Bonsai file: handles the three raw filename date formats, collapses
# duplicated frames, scales tracking to cm and decodes TTL pulses into events.

def parse_raw_date(fname):
    m=re.search(r'_(\d{2})_(\d{2})_(\d{2})_(\d{2})', fname)
    if m: return ''.join(m.groups()[:3]), m.group(4)
    m=re.search(r'_(\d{6})_(\d{2})(?:\d{4})?-', fname)
    if m: return m.group(1), m.group(2)
    m=re.search(r'_(\d{6})\d{4}-', fname)
    if m: return m.group(1), '00'
    return 'Unknown','00'

def clean_bonsai_raw(df_raw, animal_id):
    df=df_raw.rename(columns={'Item1':'TimestampRaw_Bonsai','Item2':'ROI',
        'Item3':'Ypos','Item4':'Xpos','Item5':'Event'}).copy()
    df['Timestamp_Bonsai']=df['TimestampRaw_Bonsai']-df['TimestampRaw_Bonsai'].min()
    # ── Bonsai CombineLatest de-duplication ──────────────────────────────────
    # M26/M27/M28 were recorded with a Bonsai workflow where each
    # observable (Xpos, Ypos, ROI, Event) fires independently, so every frame
    # appears 4x under one timestamp. dt then = 0 on 3 of 4 rows and Speed_cm
    # comes out inf/NaN (and 0.0 where it survives) -- the column is unusable.
    # Collapse each timestamp group to its LAST row, which carries that frame's
    # updated position, OR-ing Event across the group first so no pulse sample
    # is dropped (M26 has 69 timestamps whose Event is True on a non-last row;
    # losing them would corrupt the pulse durations the classifier keys on).
    # Idempotent -- strictly a no-op for the 21 animals already 1 row/frame.
    if df['Timestamp_Bonsai'].duplicated().any():
        _n0=len(df)
        df=df.sort_values('Timestamp_Bonsai', kind='stable')
        df['_ev']=df['Event'].astype(str).str.lower().isin(['true','1','1.0'])
        df['_ev']=df.groupby('Timestamp_Bonsai')['_ev'].transform('any')
        df=df.drop_duplicates('Timestamp_Bonsai', keep='last').reset_index(drop=True)
        df['Event']=df.pop('_ev')
        print(f"    {animal_id}: CombineLatest de-dup {_n0} -> {len(df)} rows")
    x_min,x_max=df['Xpos'].min(),df['Xpos'].max(); sf=18.0/(x_max-x_min)
    df['Xscaled']=(df['Xpos']-x_min)*sf; df['Yscaled']=(df['Ypos']-df['Ypos'].min())*sf
    dt=df['Timestamp_Bonsai'].diff()
    df['Speed_cm']=np.sqrt(df['Xscaled'].diff()**2+df['Yscaled'].diff()**2)/dt
    df.loc[df.index[0],'Speed_cm']=0.0
    df['DistFromFED']=np.sqrt((df['Xscaled']-FED_X_CM)**2+df['Yscaled']**2)
    ev=df['Event'].astype(str).str.lower().isin(['true','1','1.0']).values
    ts=df['Timestamp_Bonsai'].values; rows=[]; start=None
    for i in range(len(ev)):
        prev=ev[i-1] if i>0 else False
        if ev[i] and not prev: start=ts[i]
        if start is not None and ((not ev[i] and prev) or (i==len(ev)-1 and ev[i])):
            rows.append({'start_time':start,'duration':ts[i]-start}); start=None
    ev_df=pd.DataFrame(rows)
    if ev_df.empty: raise ValueError(f"{animal_id}: no pulse events")
    def cl(d):
        if TONE_PULSE[0]<=d<=TONE_PULSE[1]: return 'Tone'
        if DISPENSE_PULSE[0]<=d<=DISPENSE_PULSE[1]: return 'Dispense'
        if RETRIEVAL_PULSE[0]<=d<=RETRIEVAL_PULSE[1]: return 'Retrieval'
        return 'Unknown'
    ev_df['event_type']=ev_df['duration'].apply(cl)
    if get_animal_num(animal_id)<5:
        ev_df.loc[ev_df['event_type']=='Dispense','event_type']='Retrieval'
    tones=ev_df[ev_df['event_type']=='Tone'].sort_values('start_time').reset_index(drop=True)
    tones['dur_ms']=[TONE_DURATIONS[i%len(TONE_DURATIONS)] for i in range(len(tones))]
    for c in ['ToneStart_5000','ToneStart_10000','ToneStart_20000','Dispense_Bonsai','Retrieval_Bonsai']:
        df[c]=0
    for _,r in tones.iterrows():
        df.loc[(df['Timestamp_Bonsai']-r['start_time']).abs().idxmin(), f"ToneStart_{int(r['dur_ms'])}"]=1
    for _,r in ev_df[ev_df['event_type']=='Dispense'].iterrows():
        df.loc[(df['Timestamp_Bonsai']-r['start_time']).abs().idxmin(),'Dispense_Bonsai']=1
    for _,r in ev_df[ev_df['event_type']=='Retrieval'].iterrows():
        df.loc[(df['Timestamp_Bonsai']-r['start_time']).abs().idxmin(),'Retrieval_Bonsai']=1
    return df
print("clean_bonsai_raw defined (reference).")


clean_bonsai_raw defined (reference).


In [ ]:
# @title Stage A — Bonsai (canonical: clean from raw; 'scaled' loads pre-cleaned)
if BONSAI_SOURCE=='scaled':
    _b=gh_load_folder(API_SCALED, desc="Stage A: cleaned Bonsai")
    bonsai_by_animal={a:d['df'] for a,d in _b.items()}
    print(f"Loaded cleaned Bonsai for {len(bonsai_by_animal)} animals.")
elif BONSAI_SOURCE=='raw':
    _r=gh_load_folder(API_RAW, desc="Stage A: raw Bonsai")
    bonsai_by_animal={}
    for a,d in _r.items():
        try: bonsai_by_animal[a]=clean_bonsai_raw(d['df'],a)
        except Exception as e: print(f"  x {a}: {e}")
    print(f"Cleaned {len(bonsai_by_animal)} animals from raw Bonsai.")
else:
    raise ValueError("BONSAI_SOURCE must be 'scaled' or 'raw'")


Stage A: raw Bonsai:   0%|          | 0/24 [00:00<?, ?it/s]

    M28: CombineLatest de-dup 314704 -> 78649 rows
    M27: CombineLatest de-dup 458285 -> 114539 rows
  x F19: F19: no pulse events
    M26: CombineLatest de-dup 602699 -> 150641 rows
Cleaned 23 animals from raw Bonsai.


In [ ]:
# @title Stage B — photometry functions (debleach -> motion -> 25th-pctile z)
def double_exponential(t, const, amp_fast, amp_slow, tau_slow, tau_multiplier):
    tau_fast=tau_slow*tau_multiplier
    return const+amp_slow*np.exp(-t/tau_slow)+amp_fast*np.exp(-t/tau_fast)

def load_rwd_raw(download_url):
    txt=requests.get(download_url); txt.raise_for_status(); txt=txt.text
    for h in (0,1):
        d=pd.read_csv(io.StringIO(txt), header=h, low_memory=False)
        if ('TimeStamp' in d.columns or 'Time' in d.columns) and \
           ('Events' in d.columns or 'CH1-470' in d.columns or 'Fluorescence' in d.columns):
            return d
    return pd.read_csv(io.StringIO(txt), low_memory=False)

def process_photometry(df_raw):
    df=df_raw.copy()
    if 'Time' in df.columns and 'TimeStamp' not in df.columns: df=df.rename(columns={'Time':'TimeStamp'})
    if 'CH1-410' in df.columns: df=df.rename(columns={'CH1-410':'UV'})
    if 'CH1-470' in df.columns: df=df.rename(columns={'CH1-470':'Fluorescence'})
    df=df.loc[:, ~df.columns.str.contains('^Unnamed', regex=True)]
    df['TimeStampAdjusted']=df['TimeStamp']/1000 if df['TimeStamp'].max()>60000 else df['TimeStamp']
    t=df['TimeStampAdjusted']
    def _fit(sig):
        mx=np.max(sig); bounds=([0,0,0,0,0],[mx,mx,mx,36000,1]); p0=[mx/2,mx/4,mx/4,7500,0.1]
        p,_=curve_fit(double_exponential,t,sig,p0=p0,bounds=bounds,maxfev=1000)
        return double_exponential(t,*p)
    fl,uv=df['Fluorescence'],df['UV']
    fl_db=fl-_fit(fl); uv_db=uv-_fit(uv)
    slope,intercept,*_=linregress(x=uv_db,y=fl_db)
    est=intercept+slope*uv_db; corr=fl_db-est
    p25=np.percentile(corr,25); bottom=corr[corr<=p25]
    z25=(corr-bottom.mean())/bottom.std()
    df['Debleached Fluorescence']=fl_db; df['Debleached UV']=uv_db
    df['Estimated Motion']=est; df['CorrectedFluorescence']=corr
    df['Corrected_Fluorescence_Z_25']=z25
    df['CorrectedFluorescence_Z']=(corr-corr.mean())/corr.std()
    return df
print("Stage B functions defined.")


Stage B functions defined.


In [ ]:
# @title Stage C — RWD event helper (derive ToneStart / PelletGrab)
_TONE_RWD=['ToneStart_5_RWD','ToneStart_10_RWD','ToneStart_20_RWD']
_RWD_TO_CANON={'ToneStart_5_RWD':'ToneStart_5000','ToneStart_10_RWD':'ToneStart_10000',
               'ToneStart_20_RWD':'ToneStart_20000'}
def add_rwd_events(df):
    # Canonical tone markers come from the COMPLETE, exact-timed RWD record (the Bonsai
    # video can drop tones — e.g. M9 captured 11 of 21). Every animal (incl. no-Bonsai
    # F19) thus gets ToneStart_5000/10000/20000.
    for rwd_c, canon in _RWD_TO_CANON.items():
        df[canon] = df[rwd_c].astype(int) if rwd_c in df.columns else 0
    tc=[c for c in _TONE_RWD if c in df.columns]
    df['ToneStart']=(df[tc].sum(axis=1)>0).astype(int) if tc else 0
    dc=[c for c in ['Dispense_RWD','Retrieval_RWD'] if c in df.columns]
    df['PelletGrab']=(df[dc].sum(axis=1)>0).astype(int) if dc else 0
    return df
print("Stage C helper defined.")


Stage C helper defined.


In [ ]:
# @title Stage D — align Bonsai->RWD clock + interpolate (functions)
def _event_times(df, cols, tcol):
    cols=[c for c in cols if c in df.columns]
    if not cols: return np.array([])
    return np.sort(df.loc[(df[cols].sum(axis=1)>0).values, tcol].values)

def align_and_merge(photo_df, bonsai_df):
    """Align the Bonsai clock to RWD by PROXIMITY-matching shared events (robust to a
    Bonsai recording that dropped events), then interpolate tracking + snap the Bonsai
    dispense/retrieval markers onto the RWD axis. Tone markers come from RWD (Stage C)."""
    df=photo_df.copy()
    rwd_ev=_event_times(photo_df, RWD_EVENT_COLS, 'TimeStampAdjusted')
    bon_ev=_event_times(bonsai_df, BONSAI_ALIGN, 'Timestamp_Bonsai')
    if len(rwd_ev)<2 or len(bon_ev)<2: return df, None

    # ── Coarse offset: anchor a SPREAD of Bonsai events (not just the first few —
    # some recordings have spurious extra events at the start) against every RWD
    # event. Covers any offset (some Bonsai clocks are shifted by >1000 s) and is
    # robust to junk events anywhere. Pick the shift matching the most events. ──
    rr=rwd_ev[None,:]
    best_off, best_n = 0.0, -1
    _aidx=np.unique(np.linspace(0, len(bon_ev)-1, min(10, len(bon_ev))).astype(int))
    for ba in bon_ev[_aidx]:
        for rt in rwd_ev:
            off=rt-ba
            d=np.min(np.abs((bon_ev[:,None]+off)-rr), axis=1)
            n=int((d<0.5).sum())
            if n>best_n: best_n, best_off = n, off

    # ── Proximity match + linear (offset+drift) fit on matched pairs only ────────
    shifted=bon_ev+best_off
    j=np.argmin(np.abs(shifted[:,None]-rr), axis=1)
    keep=np.abs(shifted-rwd_ev[j])<1.0
    mb, mr = bon_ev[keep], rwd_ev[j[keep]]
    if len(mb)<3: return df, None
    slope,intercept,r,_,_=linregress(mb, mr)

    # ── Refine: re-match using the fitted line (accounts for drift) and refit ────
    for _ in range(5):
        pred=slope*bon_ev+intercept
        j2=np.argmin(np.abs(pred[:,None]-rr), axis=1)
        keep2=np.abs(pred-rwd_ev[j2])<1.0
        if int(keep2.sum())<3: break
        mb2, mr2 = bon_ev[keep2], rwd_ev[j2[keep2]]
        s2,i2,r2,_,_=linregress(mb2, mr2)
        converged = abs(s2-slope)<1e-9 and abs(i2-intercept)<1e-6
        mb, mr, slope, intercept, r = mb2, mr2, s2, i2, r2
        if converged: break

    # Too few matched events to trust a slope -> offset-only (assume no drift).
    if len(mb) < 8:
        slope = 1.0; intercept = float(np.median(mr - mb)); r = float('nan')

    bcorr=bonsai_df['Timestamp_Bonsai'].values*slope+intercept
    rwd_t=df['TimeStampAdjusted'].values
    for c in BONSAI_CONT:
        if c in bonsai_df.columns:
            f=interp1d(bcorr, bonsai_df[c].values, kind='linear', bounds_error=False, fill_value=np.nan)
            df[c]=f(rwd_t)
    for c in BONSAI_SNAP:
        if c in bonsai_df.columns:
            arr=np.zeros(len(df),dtype=int)
            for t in bcorr[bonsai_df[c].values==1]: arr[np.argmin(np.abs(rwd_t-t))]=1
            df[c]=arr
    info={'offset':intercept,'slope':slope,'r':r,'n_match':int(len(mb)),
          'n_rwd_ev':len(rwd_ev),'n_bon_ev':len(bon_ev),'drift_s_per_hr':(1-slope)*3600,
          'bcorr':bcorr,'rwd_ev':rwd_ev,'bon_ev':bon_ev,'mb':mb,'mr':mr,'best_off':best_off}
    return df, info
print("Stage D functions defined.")


Stage D functions defined.


In [ ]:
# @title Stage E — position normalization (function)
def normalize_positions(df, animal_id):
    df=df.copy(); group=CONDITION_MAP.get(animal_id,'?')
    rec={'Animal':animal_id,'Group':group,'Status':'NO_BONSAI','FED_x_px':np.nan,
         'FED_y_px':np.nan,'scale_cm_per_px':np.nan,'R2':np.nan,'MAE_cm':np.nan,'n_retrieval':0}
    if 'Xpos' not in df.columns or 'Ypos' not in df.columns: return df, rec
    x_raw=pd.to_numeric(df['Xpos'],errors='coerce').values
    y_raw=pd.to_numeric(df['Ypos'],errors='coerce').values
    anchor=ANCHOR_COL if ANCHOR_COL in df.columns else 'DistFromFED'
    dist=pd.to_numeric(df[anchor],errors='coerce').values
    valid_xy=~(np.isnan(x_raw)|np.isnan(y_raw))
    if 'Retrieval_Bonsai' not in df.columns: rec['Status']='NO_RETRIEVAL'; return df, rec
    retr=(pd.to_numeric(df['Retrieval_Bonsai'],errors='coerce')==1).values & valid_xy
    rec['n_retrieval']=int(retr.sum())
    dr=dist[retr]; near=retr.copy(); near[retr]=(dr<NEAR_FED_CM)|np.isnan(dr)
    if near.sum()<3: near=retr
    fed_x=float(np.nanmedian(x_raw[near])); fed_y=float(np.nanmedian(y_raw[near]))
    rec['FED_x_px']=round(fed_x,1); rec['FED_y_px']=round(fed_y,1)
    xc=x_raw-fed_x; yc=y_raw-fed_y
    speed=pd.to_numeric(df['Speed_cm'],errors='coerce').values if 'Speed_cm' in df.columns else np.zeros(len(df))
    vd=valid_xy & ~np.isnan(dist) & (dist>=0) & (speed<SPEED_MAX)
    if vd.sum()<MIN_REG_PTS: vd=valid_xy & ~np.isnan(dist) & (dist>=0)
    if vd.sum()<10: rec['Status']='REGRESSION_FAILED'; return df, rec
    d_px=np.sqrt(xc[vd]**2+yc[vd]**2); d_cm=dist[vd]
    denom=np.sum(d_px**2); scale=float(np.sum(d_px*d_cm)/denom) if denom>0 else np.nan
    ss_res=np.sum((d_cm-scale*d_px)**2); ss_tot=np.sum((d_cm-d_cm.mean())**2)
    r2=float(1-ss_res/ss_tot) if ss_tot>0 else np.nan
    rec['scale_cm_per_px']=round(scale,5); rec['R2']=round(r2,4)
    x_cm=xc*scale; y_cm=yc*scale
    vcm=valid_xy & np.isfinite(x_cm) & np.isfinite(y_cm)
    if vcm.sum()<10: rec['Status']='PCA_FAILED'; return df, rec
    pca=PCA(n_components=2); pca.fit(np.column_stack([x_cm[vcm],y_cm[vcm]]))
    rot=pca.components_[[1,0],:]; xyr=np.column_stack([x_cm,y_cm])@rot.T
    x_norm,y_norm=xyr[:,0],xyr[:,1]
    if np.nanmedian(y_norm[valid_xy])<0: y_norm=-y_norm
    dist_re=np.sqrt(x_norm**2+y_norm**2)
    mae=float(np.nanmean(np.abs(dist_re[vd]-dist[vd]))); rec['MAE_cm']=round(mae,3)
    status='OK'
    if r2<R2_FLAG: status=f'LOW_R2({r2:.3f})'
    elif mae>MAE_FLAG: status=f'HIGH_MAE({mae:.2f}cm)'
    rec['Status']=status
    if ANCHOR_COL not in df.columns: df[ANCHOR_COL]=df['DistFromFED']
    df['Xpos_cm']=np.where(valid_xy,x_norm,np.nan)
    df['Ypos_cm']=np.where(valid_xy,y_norm,np.nan)
    if OVERWRITE_DISTFROMFED: df['DistFromFED']=np.sqrt(df['Xpos_cm']**2+df['Ypos_cm']**2)
    return df, rec
print("Stage E function defined.")

# ── Stage E2 — tracker losses ("phantom points") ─────────────────────────────
# When the video tracker loses the mouse it can latch onto a fixed
# spot (a corner, a reflection) and jump there and back. Those samples are not the
# mouse: M13 sat on a corner 18 cm from the FED for ~26 min, recorded far from the
# feeder while photometry shows it retrieving pellets. Detect and blank them.
#   jump    : > JUMP_CM between consecutive samples (> 150 cm/s at 15 Hz; impossible)
#   phantom : a 2 cm grid cell where >= PHANTOM_MIN_LANDINGS jumps land, at least
#             PHANTOM_MIN_FED_CM from the FED (mice really do sit at the feeder)
#   lost    : every jump sample, plus each stretch within PHANTOM_RADIUS_CM of a
#             phantom that was entered by a jump or sat frozen there for >= 1 s
# Lost samples -> NaN in Xpos_cm / Ypos_cm / DistFromFED / Speed_cm; flagged in
# TrackingLost. Raw pixel Xpos / Ypos are left untouched for provenance.
JUMP_CM = 10.0; PHANTOM_MIN_LANDINGS = 4; PHANTOM_RADIUS_CM = 2.5; PHANTOM_MIN_FED_CM = 5.0

def mask_phantom_tracking(df, rec):
    if 'Xpos_cm' not in df.columns:
        rec['tracking_lost_pct'] = np.nan; return df, rec
    df = df.copy()
    x = df['Xpos_cm'].values; y = df['Ypos_cm'].values
    n1s = int(round(1 / np.median(np.diff(df['TimeStampAdjusted'].values))))
    step = np.r_[0, np.hypot(np.diff(x), np.diff(y))]
    J = np.where(step > JUMP_CM)[0]
    lost = np.zeros(len(df), bool); lost[J] = True
    phantoms = []
    if len(J):
        land = np.c_[x[J], y[J]]; cell = np.floor(land / 2).astype(int)
        _, inv, cnt = np.unique(cell, axis=0, return_inverse=True, return_counts=True)
        for k in np.where(cnt >= PHANTOM_MIN_LANDINGS)[0]:
            c = land[inv.ravel() == k].mean(0)
            if np.hypot(*c) >= PHANTOM_MIN_FED_CM: phantoms.append(c)
    frozen = step < 0.05
    for c in phantoms:
        near = np.hypot(x - c[0], y - c[1]) < PHANTOM_RADIUS_CM
        e = np.diff(np.r_[0, near.astype(int), 0]); on, off = np.where(e == 1)[0], np.where(e == -1)[0]
        for a, b in zip(on, off):
            if (np.abs(J - a) <= 2).any() or (b - a >= n1s and frozen[a:b].mean() > .8):
                lost[a:b] = True
    df['TrackingLost'] = lost.astype(int)
    for col in ('Xpos_cm', 'Ypos_cm', 'DistFromFED', 'Speed_cm'):
        if col in df.columns: df.loc[lost, col] = np.nan
    rec['tracking_lost_pct'] = round(100 * lost.mean(), 2); rec['n_phantoms'] = len(phantoms)
    if lost.mean() > 0.01:
        print(f"    {rec.get('Animal', '?')}: tracking lost {100 * lost.mean():.1f}% "
              f"({lost.sum() / n1s / 60:.1f} min, {len(phantoms)} phantom point(s)) -> masked")
    return df, rec
print("Stage E2 (tracker-loss mask) defined.")



In [ ]:
# @title Per-animal QC figure
def qc_figure(aid, df, info, rec, out_png):
    grp=CONDITION_MAP.get(aid,'?')
    fig,ax=plt.subplots(2,3,figsize=(16,7))
    fig.suptitle(f'{aid} [{grp}]  norm:{rec.get("Status","-")}  R2={rec.get("R2","-")}  MAE={rec.get("MAE_cm","-")}cm',
                 fontsize=12, fontweight='bold')
    t=df['TimeStampAdjusted'].values
    # 1 raw + fits
    ax[0,0].plot(t, df['Fluorescence'],'g',lw=.4,alpha=.7,label='Fluor')
    ax[0,0].plot(t, df['Fluorescence']-df['Debleached Fluorescence'],'r',lw=.8,label='exp fit')
    ax[0,0].set_title('Raw fluorescence + debleach fit',fontsize=9); ax[0,0].legend(fontsize=7)
    # 2 z-score trace + events
    ax[0,1].plot(t, df['Corrected_Fluorescence_Z_25'],color='dimgray',lw=.4)
    for c,col in [('ToneStart_5000','#E05C5C'),('ToneStart_10000','#E09C3C'),
                  ('ToneStart_20000','#3C7BE0'),('Retrieval_Bonsai','purple')]:
        if c in df.columns:
            for tt in t[df[c].values==1]: ax[0,1].axvline(tt,color=col,lw=.5,alpha=.5)
    ax[0,1].set_title('Z-score (25%ile) + Bonsai events',fontsize=9)
    # 3 motion correction scatter
    ax[0,2].scatter(df['Debleached UV'][::20], df['Debleached Fluorescence'][::20],s=1,alpha=.2,color='gray')
    ax[0,2].set_title('Motion fit (UV vs Fluor debleached)',fontsize=9)
    # 4 alignment drift
    if info is not None:
        mb, mr = info['mb'], info['mr']
        resid = mr - (info['slope']*mb + info['offset'])
        ax[1,0].plot(range(len(mb)), resid, 'o-', ms=3)
        ax[1,0].axhline(0,color='k',lw=.6,ls='--')
        ax[1,0].set_title(f"Align resid (n={len(mb)})  slope={info['slope']:.5f}  ({info['drift_s_per_hr']:.2f}s/hr)",fontsize=9)
        ax[1,0].set_xlabel('matched event #'); ax[1,0].set_ylabel('RWD - pred (s)')
    else:
        ax[1,0].text(.5,.5,'no Bonsai',ha='center',transform=ax[1,0].transAxes)
    # 5 raw trajectory + FED
    if 'Xpos' in df.columns:
        ax[1,1].plot(df['Xpos'],df['Ypos'],'k-',lw=.2,alpha=.4)
        if not np.isnan(rec.get('FED_x_px',np.nan)):
            ax[1,1].scatter([rec['FED_x_px']],[rec['FED_y_px']],color='red',s=50,zorder=5,label='FED est')
        ax[1,1].set_title('Raw trajectory (px) + FED',fontsize=9); ax[1,1].set_aspect('equal'); ax[1,1].legend(fontsize=7)
    # 6 normalized cm
    if 'Xpos_cm' in df.columns:
        ax[1,2].plot(df['Xpos_cm'],df['Ypos_cm'],'k-',lw=.2,alpha=.4)
        ax[1,2].scatter([0],[0],color='red',s=50,zorder=5,label='FED (0,0)')
        ax[1,2].set_title(f"Normalized cm  scale={rec.get('scale_cm_per_px','-')}",fontsize=9)
        ax[1,2].set_aspect('equal'); ax[1,2].legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(out_png, dpi=110, bbox_inches='tight'); plt.close()
print("QC figure function defined.")


QC figure function defined.


In [ ]:
# @title Stage F — batch all animals (skip-existing) + QC + verify + save

# Animal universe = the RWD_Fluorescence files (every animal has photometry).
rwd_files = gh_list(API_RWD_FLUOR)
rwd_index = {get_animal_id(f['name']): f for f in rwd_files if get_animal_id(f['name'])}

# Animals whose processed data is ALREADY committed to GitHub -- the skip set.
# Keyed by animal ID, so the M13 filename date quirk can't cause a false miss.
processed_on_github = set()
if SKIP_EXISTING:
    processed_on_github = {get_animal_id(f['name']) for f in gh_list(API_MERGED)
                           if get_animal_id(f['name'])}

# Legacy merged data (pinned commit) for verification -- see LEGACY_REF.
merged_index = {}
if VERIFY_VS_EXISTING:
    merged_index = {get_animal_id(f['name']): f for f in gh_list(API_MERGED_REF)
                    if get_animal_id(f['name'])}

def out_name_for(aid):
    base = rwd_index[aid]['name'].replace('.csv','')
    return f"{base}_merged.csv" if aid in bonsai_by_animal else f"{base}_merged_NOBONSAIDATA.csv"

qc_rows, verify_rows, skipped = [], [], []
for aid in tqdm(sorted(rwd_index, key=lambda s:(s[0], get_animal_num(s))), desc="Stage F: animals"):
    out_path = f"{OUT_DIR}/{out_name_for(aid)}"
    if SKIP_EXISTING and aid in processed_on_github:
        skipped.append(aid); continue
    try:
        df = process_photometry(load_rwd_raw(rwd_index[aid]['download_url']))
        df = add_rwd_events(df)
        info = None
        rec = {'Animal':aid,'Group':CONDITION_MAP.get(aid,'?'),'Status':'NO_BONSAI',
               'R2':np.nan,'MAE_cm':np.nan,'n_retrieval':0,'scale_cm_per_px':np.nan,
               'FED_x_px':np.nan,'FED_y_px':np.nan}
        if aid in bonsai_by_animal:
            df, info = align_and_merge(df, bonsai_by_animal[aid])
            df, rec  = normalize_positions(df, aid)
            df, rec  = mask_phantom_tracking(df, rec)   # Stage E2
        df.to_csv(out_path, index=False)
        qc_figure(aid, df, info, rec, f"{QC_DIR}/qc_{rec['Group']}_{aid}.png")
        row = dict(rec); row['align_r']=None if info is None else round(info['r'],4)
        row['drift_s_per_hr']=None if info is None else round(info['drift_s_per_hr'],3)
        qc_rows.append(row)

        if VERIFY_VS_EXISTING and aid in merged_index:
            ref = gh_read(merged_index[aid]['download_url'])
            v={'Animal':aid}
            if 'Corrected_Fluorescence_Z_25' in ref.columns:
                a=df['Corrected_Fluorescence_Z_25'].values; b=ref['Corrected_Fluorescence_Z_25'].values
                n=min(len(a),len(b)); v['z_corr']=round(float(np.corrcoef(a[:n],b[:n])[0,1]),5)
            for c in ['ToneStart_5000','ToneStart_10000','ToneStart_20000','Dispense_Bonsai','Retrieval_Bonsai']:
                if c in df.columns and c in ref.columns:
                    v[c]=f"{int(df[c].sum())}/{int(ref[c].sum())}"
            verify_rows.append(v)
    except Exception as e:
        print(f"  x {aid}: {e}")

print(f"\nProcessed {len(qc_rows)}  |  skipped (already on GitHub): {len(skipped)} {skipped}")
if skipped and not qc_rows:
    print("Nothing to do -- every animal's processed data is already committed to the repo.\n"
          "Set SKIP_EXISTING=False in the options cell to rebuild them all from raw.")
if qc_rows:
    print("\n── Normalization / alignment QC ──")
    print(pd.DataFrame(qc_rows).reindex(columns=['Animal','Group','Status','R2','MAE_cm','n_retrieval','align_r','drift_s_per_hr','tracking_lost_pct']).to_string(index=False))
    pd.DataFrame(qc_rows).to_csv(f"{OUT_DIR}/preprocessing_qc.csv", index=False)
if verify_rows:
    print(f"\n── Verify vs reference processed data @ {LEGACY_REF} (z corr; event counts mine/ref) ──")
    print(pd.DataFrame(verify_rows).to_string(index=False))
print(f"\nWrote merged CSVs + QC to {OUT_DIR}")


In [ ]:
# @title Download the results (zip -> your machine)
# Colab-local output does not survive the runtime, so pull it down before closing.
# Commit the merged CSVs to Photometry/VariableTone/RWD/RWD_Processed/
# yourself -- the notebook never writes to the repo.
import shutil
from google.colab import files

if os.path.isdir(OUT_DIR) and os.listdir(OUT_DIR):
    _zip = shutil.make_archive('/content/VT_Preprocessing_out', 'zip', OUT_DIR)
    print(f"Zipped -> {_zip}  ({os.path.getsize(_zip)/1e6:.0f} MB)")
    files.download(_zip)
else:
    print(f"{OUT_DIR} is empty (everything was skipped) -- nothing to download.")
